# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR^2 Clinicopathological Colorectal Cancer Survivor dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Install `mlcroissant` if not already installed
!pip install -U mlcroissant

## 1. Data Loading

Load main Croissant metadata and fetch records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Dataset.metadata is a DatasetMetadata object; to_json() gives a dict.
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.date_published}")
print(f"Croissant Version: {dataset.metadata.conforms_to}")
print(f"Number of record sets: {len(dataset.metadata.record_sets)}")

## 2. Data Overview
Let's review the available record sets, the fields inside them, and their Croissant `@id` identifiers.

In [ ]:
# List all record sets and their fields using their @id
from pprint import pprint

record_sets = dataset.metadata.record_sets
print(f"Dataset contains {len(record_sets)} record set(s):\n")
record_set_ids = []

for rs in record_sets:
    print(f"- RecordSet: {rs.name} (@id: {rs.id})")
    record_set_ids.append(rs.id)
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print("")

### View Example Records from Each Record Set

Records from a record set are accessed via its `@id`. Let's print the first 2 records from each found record set:

In [ ]:
# Show example records from each record set using @id
for rs_id in record_set_ids:
    print(f"\nRecordSet @id: {rs_id}")
    records = dataset.records(record_set=rs_id)
    for i, rec in enumerate(records):
        if i >= 2:
            break
        pprint(rec)
    if i == 0:
        print("  (No records loaded)")

## 3. Data Extraction

Extract all records for each record set into dataframes for structured analyses. Entities are always referenced by their `@id`.

In [ ]:
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {rs_id} (shape: {df.shape})")
    else:
        print(f"No records found for RecordSet @id: {rs_id}, skipping.")

if dataframes:
    # Pick the first available record set for demonstration
    main_rs_id = next(iter(dataframes))
    print(f"\nColumns in DataFrame for RecordSet @id '{main_rs_id}':")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

We now demonstrate common data processing steps (e.g., filtering, normalization, grouping) by field `@id`, and showcase each step.

**Note:** Please update `numeric_field_id` and `group_field_id` to real column `@id`s as found above for your target analysis.

In [ ]:
# Example: Filtering, normalizing, and grouping - update these for your dataset!
# Use the actual @id of the record set and fields from previous step

record_set_id = main_rs_id             # Use main record set loaded above
df = dataframes[record_set_id]

# Example: pick likely numeric and group-by fields by id (replace by real ids if known)
possible_numeric_ids = [col for col in df.columns if any(x in col.lower() for x in ["age", "interval", "years", "count", "months", "number"])]
possible_group_ids  = [col for col in df.columns if any(x in col.lower() for x in ["sex", "gender", "site", "location", "status", "histology"])]

print("Numeric-like fields found by @id for EDA:", possible_numeric_ids)
print("Groupable/categorical fields by @id:", possible_group_ids)

# For demonstration, select the first found; if not, fallback to any column
numeric_field_id = possible_numeric_ids[0] if possible_numeric_ids else df.columns[0]
group_field_id = possible_group_ids[0] if possible_group_ids else df.columns[1] if len(df.columns) > 1 else df.columns[0]

print(f"\nUsing numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}\n")

# Filtering: keep only records with positive numeric field value greater than a threshold (e.g. 10)
try:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
except:
    pass
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the group field and show aggregated mean statistics
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Let's plot histogram of the selected numeric field, and a boxplot grouped by the chosen categorical field. (Update field `@id`s for most appropriate visualization in your data context.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot by group field
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

You have successfully loaded and explored the FAIR^2 colorectal cancer survivor dataset with `mlcroissant`. This included extracting record sets and fields using their Croissant `@id`, filtering, normalizing, grouping, and visualizing data—all with explicit, reproducible references to the Croissant schema. Remember, for robust domain insights and publication-ready analyses, tailor the field selections to the specific medical, demographic, and molecular features most relevant to your research.